# 13장 텐서플로를 사용한 데이터 적재와 전처리

## tf.data 
> 대용량 데이터셋에서 텐서플로 모델을 훈련할 때 사용하는 데이터 로드 및 전처리 API

### tf.data의 장점 
- 효율적인 데이터 로드 및 전처리 가능 
- 멀티스레드 및 큐를 이용하여 여러 파일을 동시에 처리 가능 
- 샘플 셔플링 및 배치 작업 가능 
- GPU 및 TPU가 훈련 및 데이터 배치 처리 하는 동안 CPU 코어를 통해 데이터 배치 로드 및 전처리 가능 
- 메모리보다 큰 데이터 셋 처리 가능 
- 훈련 속도 향상 

### 지원 포맷 
- 텍스트 파일 
- 이진 파일
- TFRecord 

### 케라스의 전처리층 
- 모델을 제품 환경에 배포 시, 다른 전처리 코드 추가 필요 없음 
- 훈련/서빙 차이의 위험 제거 
- 동일한 전처리 코드 구현 필요 없음 → 불일치 위험 감소 

-----

# 1. 데이터 API 

## tf.data.Dataset
- 데이터 항목의 시퀀스를 나타냄 
- 데이터셋에서는 텐서 튜플, 딕셔너리 (이름/텐서 쌍), 중첩된 튜플 및 딕셔너리도 가능 
  - 슬라이싱 시, 데이터셋은 구조를 유지하며 안의 텐서만 슬라이싱

### 매소드 
- `from_tensor_slices()` : 텐서를 받아 첫 번째 차원을 따라 원소가 아이템으로 표현되는 `tf.data.Dataset` 생성 

- 예시 : 텐서 데이터셋 생성

In [18]:
import tensorflow as tf

### 임의의 데이터 텐서 생성
X = tf.range(10)

### dataset 생성
dataset = tf.data.Dataset.from_tensor_slices(X)

### dataset 출력
print("dataset : ", dataset)

print("dataset 아이템 : ")
for item in dataset:
    print(item)


dataset :  <_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>
dataset 아이템 : 
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


- 예시 : 텐서만 슬라이싱

In [19]:
X_nested = {"a":([1,2,3],[4,5,6]), "b":[7,8,9]}
dateset = tf.data.Dataset.from_tensor_slices(X_nested)

for item in dateset:
    print(item)

{'a': (<tf.Tensor: shape=(), dtype=int32, numpy=1>, <tf.Tensor: shape=(), dtype=int32, numpy=4>), 'b': <tf.Tensor: shape=(), dtype=int32, numpy=7>}
{'a': (<tf.Tensor: shape=(), dtype=int32, numpy=2>, <tf.Tensor: shape=(), dtype=int32, numpy=5>), 'b': <tf.Tensor: shape=(), dtype=int32, numpy=8>}
{'a': (<tf.Tensor: shape=(), dtype=int32, numpy=3>, <tf.Tensor: shape=(), dtype=int32, numpy=6>), 'b': <tf.Tensor: shape=(), dtype=int32, numpy=9>}


---

## 연쇄 변환

> 연쇄 변환은 tf.data.Dataset의 메서드들이 원본 데이터셋을 변경하지 않고 새로운 데이터셋을 반환하는 특성을 활용하여, 여러 변환 메서드를 연속적으로 연결하여 데이터 파이프라인을 구축하는 방식

### 메소드
  - `repeat()` : 원본 데이터 셋의 아이템을 반복하는 데이터셋 반환 
    - 메모리에서 반복하는 것은 아님 
  - `batch()` : 새로운 데이터셋을 미니 배치로 생성
    - `drop_remainder=True` : 모든 배치를 동일한 크기로 맞춤 
  - `map()` : 아이템 변환 및 전처리 가능 
    - `num_parallel_calls` : 실행 할 스레드 수 결정 
    - `tf.data.AUTOTUNE` 지정 가능 
  - `filter()` : 데이터셋 필터링
  - `take()` : 원하는 데이터만 조회 

- 연쇄 변환 예시 
  - 0~9까지 텐서 생성 
  - repeat로 3번 반복 
  - batch 로 7개의 미니 배치 생성 
  - 데이터셋의 아이템 순회

In [20]:
dataset = tf.data.Dataset.from_tensor_slices(tf.range(10))
dataset = dataset.repeat(3).batch(7)
for item in dataset:
    print(item)

tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)
tf.Tensor([8 9], shape=(2,), dtype=int32)


- 예시 : map을 통한 아이템 변환
  - 모든 아이템에 2를 곱함 

In [21]:
## x : 하나의 배치 - 하나의 배치를 x*2 함 
dataset = dataset.map(lambda x: x * 2)

for item in dataset:
    print(item)

tf.Tensor([ 0  2  4  6  8 10 12], shape=(7,), dtype=int32)
tf.Tensor([14 16 18  0  2  4  6], shape=(7,), dtype=int32)
tf.Tensor([ 8 10 12 14 16 18  0], shape=(7,), dtype=int32)
tf.Tensor([ 2  4  6  8 10 12 14], shape=(7,), dtype=int32)
tf.Tensor([16 18], shape=(2,), dtype=int32)


- 예시 : filter를 통한 아이템 필터링
  - 아이템의 합이 50 이상인 경우만 필터링

In [22]:
dataset = dataset.filter(lambda x: tf.reduce_sum(x) > 50)

for item in dataset:
    print(item)

tf.Tensor([14 16 18  0  2  4  6], shape=(7,), dtype=int32)
tf.Tensor([ 8 10 12 14 16 18  0], shape=(7,), dtype=int32)
tf.Tensor([ 2  4  6  8 10 12 14], shape=(7,), dtype=int32)


- 예시 : take를 통한 데이터 조회

In [23]:
for item in dataset.take(2):
    print(item)

tf.Tensor([14 16 18  0  2  4  6], shape=(7,), dtype=int32)
tf.Tensor([ 8 10 12 14 16 18  0], shape=(7,), dtype=int32)


## 데이터 셔플링 
> 데이터셋의 샘플 순서를 무작위로 섞는 방법
- 경사하강법은 훈련세트에 있는 샘플이 독립적이고 동일한 분포일 때 최고 성능 발휘 
  - 데이터셋을 섞으면 훈련 과정에서 모델이 더 일반화된 패턴을 학습할 수 있음

- 버퍼의 크기 선정이 중요 
  - 버퍼의 크기를 충분하게 크게 해야 셔플링 효과 감소하지 않음 
  - 보유한 메모리 보다 크기를 넘지 않아야함


 - `대규모 데이터셋`의 경우 버퍼가 데이터셋에 비해 작음 -> 버퍼 방식이 충분하지 않음 
   - 원본 데이터셋 자체를 섞는 방법 사용 
   - 에포크 마다 다시 한 번 섞기 때문에 셔플링 성능 상승 



### 매소드 
- `tf.data.Dataset.shuffle()` : 데이터셋의 샘플 순서를 무작위로 섞음
  - `buffer_size` : 버퍼의 사이즈 (샘플을 무작위로 섞기 위해 사용하는 버퍼의 크기) 
  - 과정
    - 새로운 아이템 요청 
    - → 버퍼에서 랜덤하게 하나 선택 
    - → 새로운 아이템 추출 후 버퍼 채움 
    - → 모든 아이템 사용 할 때까지 반복 
    - → 버퍼가 비워질 때까지 계속 랜덤 아이템 반환 

- 예시 : 셔플링

In [24]:
dataset = tf.data.Dataset.range(10).repeat(2)
dataset = dataset.shuffle(buffer_size=4, seed=42).batch(7)
for item in dataset:
    print(item)

tf.Tensor([1 4 2 3 5 0 6], shape=(7,), dtype=int64)
tf.Tensor([9 8 2 0 3 1 4], shape=(7,), dtype=int64)
tf.Tensor([5 7 9 6 7 8], shape=(6,), dtype=int64)


## 여러 파일에서 한 줄씩 번갈아 읽기 (인터리브)

  ### 매소드 
  - `tf.data.Dataset.list_files()` : 파일 경로를 섞은 데이터셋 반환 
    - `shuffle=False` : 섞는걸 원하지 않는 경우 
  - `interleave()` : 한 줄씩 번갈아가며 읽기 
    - 병렬화 사용하지 않음 
    - 병렬을 원하는 경우 `num_parallel_calls`로 스레드 설정 
  - `skip()` : 특정 열 읽기 건너뛰기 


- 예시 캘리포니아 주택 데이터셋을 이용한 예시
  - 데이터 셋 로드 
  - 세트 분할  
  - 각 세트를 CSV 파일로 저장 
  - 해당 파일에서 한 줄씩 읽기 

- 데이터 로드 및 세트 분할

In [25]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

### 데이터 세트 로드 
housing = fetch_california_housing()

### 훈련 - 테스트 세트 분할
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target.reshape(-1, 1), random_state=42)

### 훈련 - 검증 세트 분할
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

- CSV 파일 분할 (20개)

In [ ]:
import numpy as np
from pathlib import Path

### CSV 파일 저장 함수 
def save_to_csv_files(data, name_prefix, header=None, n_parts=10):
    housing_dir = Path() / "datasets" / "housing"
    housing_dir.mkdir(parents=True, exist_ok=True)
    filename_format = "my_{}_{:02d}.csv"

    filepaths = []
    m = len(data)
    chunks = np.array_split(np.arange(m), n_parts)
    for file_idx, row_indices in enumerate(chunks):
        part_csv = housing_dir / filename_format.format(name_prefix, file_idx)
        filepaths.append(str(part_csv))
        with open(part_csv, "w") as f:
            if header is not None:
                f.write(header)
                f.write("\n")
            ### 핸즈온 머신러닝에 repr(col)로 되어 있음 -> 주의 필요
            for row_idx in row_indices:
                f.write(",".join([str(col) for col in data[row_idx]]))
                f.write("\n")
    return filepaths

train_data = np.c_[X_train, y_train]
valid_data = np.c_[X_valid, y_valid]
test_data = np.c_[X_test, y_test]
header_cols = housing.feature_names + ["MedianHouseValue"]
header = ",".join(header_cols)

## CSV 파일로 저장
train_filepaths = save_to_csv_files(train_data, "train", header, n_parts=20)
valid_filepaths = save_to_csv_files(valid_data, "valid", header, n_parts=10)
test_filepaths = save_to_csv_files(test_data, "test", header, n_parts=10)

- 파일 체크 및 파일 경로 체크 

In [60]:
### 파일 줄 체크 
print("".join(open(train_filepaths[0]).readlines()[:4]))

### 파일 경로 체크 
print("train_filepaths : ", train_filepaths)

MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedianHouseValue
3.5214,15.0,3.0499445061043287,1.106548279689234,1447.0,1.6059933407325193,37.63,-122.43,1.442
5.3275,5.0,6.490059642147117,0.9910536779324056,3464.0,3.4433399602385686,33.69,-117.39,1.687
3.1,29.0,7.5423728813559325,1.5915254237288134,1328.0,2.2508474576271187,38.44,-122.98,1.621

train_filepaths :  ['datasets\\housing\\my_train_00.csv', 'datasets\\housing\\my_train_01.csv', 'datasets\\housing\\my_train_02.csv', 'datasets\\housing\\my_train_03.csv', 'datasets\\housing\\my_train_04.csv', 'datasets\\housing\\my_train_05.csv', 'datasets\\housing\\my_train_06.csv', 'datasets\\housing\\my_train_07.csv', 'datasets\\housing\\my_train_08.csv', 'datasets\\housing\\my_train_09.csv', 'datasets\\housing\\my_train_10.csv', 'datasets\\housing\\my_train_11.csv', 'datasets\\housing\\my_train_12.csv', 'datasets\\housing\\my_train_13.csv', 'datasets\\housing\\my_train_14.csv', 'datasets\\housing\\my_train_15.csv'

- 파일 경로 데이터셋 생성 

In [61]:
filepath_dataset  = tf.data.Dataset.list_files(train_filepaths,seed=42)

### 셔플링 확인 
for filepath in filepath_dataset:
    print(filepath)

tf.Tensor(b'datasets\\housing\\my_train_05.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_16.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_01.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_17.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_00.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_14.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_10.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_02.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_12.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_19.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_07.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_09.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_13.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_15.csv', sh

- 인터리브 실행 

In [62]:
n_readers = 5
dataset = filepath_dataset.interleave(
    ### 첫번째 열 건너뜀
    lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
    cycle_length=n_readers)

#### 데이터셋 확인
for line in dataset.take(5):
    print(line)

tf.Tensor(b'4.5909,16.0,5.475877192982456,1.0964912280701755,1357.0,2.9758771929824563,33.63,-117.71,2.418', shape=(), dtype=string)
tf.Tensor(b'2.4792,24.0,3.4547038327526134,1.1341463414634145,2251.0,3.921602787456446,34.18,-118.38,2.0', shape=(), dtype=string)
tf.Tensor(b'4.2708,45.0,5.121387283236994,0.953757225433526,492.0,2.8439306358381504,37.48,-122.19,2.67', shape=(), dtype=string)
tf.Tensor(b'2.1856,41.0,3.7189873417721517,1.0658227848101265,803.0,2.0329113924050635,32.76,-117.12,1.205', shape=(), dtype=string)
tf.Tensor(b'4.1812,52.0,5.701388888888889,0.9965277777777778,692.0,2.4027777777777777,33.73,-118.31,3.215', shape=(), dtype=string)


## 데이터 전처리 

- 예시 : 캘리 포니아 주택 데이터 데이터 스케일 조정 
  - 각 특성의 평균과 표준편차 계산
  - 전처리 진행

In [63]:
###  평균 및 표준 편차 계산 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

### 평균 및 표준편차 선언 
X_mean, X_std = scaler.mean_, scaler.scale_

n_inputs = 8 

### CSV에서 한 줄씩 읽어와 파싱 
def parse_csv_line(line):
    defs = [0.] * n_inputs + [tf.constant([], dtype=tf.float32)]
    fields = tf.io.decode_csv(line, record_defaults=defs)
    return tf.stack(fields[:-1]), tf.stack(fields[-1:])

### 스칼라 텐서 리스트 반환 
def preprocess(line):
    x, y = parse_csv_line(line)
    return (x - X_mean) / X_std, y

- 전처리 함수 테스트

In [64]:
preprocess(b'4.2083,44.0,5.3232,0.9171,846.0,2.3370,37.47,-122.2,2.782')

(<tf.Tensor: shape=(8,), dtype=float32, numpy=
 array([ 0.16579159,  1.216324  , -0.05204564, -0.39215982, -0.5277444 ,
        -0.2633488 ,  0.8543046 , -1.3072058 ], dtype=float32)>,
 <tf.Tensor: shape=(1,), dtype=float32, numpy=array([2.782], dtype=float32)>)

## 데이터 적재와 전처리 합치기 
- 지금까지 언급된 함수 + 다른 헬퍼 함수를 합쳐 하나의 함수로 구성 

- 예시 : csv를 읽고 데이터 셋으로 변환 함수

In [65]:
def csv_reader_dataset(filepaths, n_readers=5, n_read_threads=None,
                       n_parse_threads=5, shuffle_buffer_size=10_000, seed=42,
                       batch_size=32):
    dataset = tf.data.Dataset.list_files(filepaths, seed=seed)
    dataset = dataset.interleave(
        lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
        cycle_length=n_readers, num_parallel_calls=n_read_threads)
    dataset = dataset.map(preprocess, num_parallel_calls=n_parse_threads)
    dataset = dataset.shuffle(shuffle_buffer_size, seed=seed)
    return dataset.batch(batch_size).prefetch(1)

- 샘플 배치 출력

In [66]:
example_set = csv_reader_dataset(train_filepaths, batch_size=3)
for X_batch, y_batch in example_set.take(2):
    print("X =", X_batch)
    print("y =", y_batch)
    print()

X = tf.Tensor(
[[-1.3957452  -0.04940685 -0.22830808  0.22648273  2.2593622   0.35200632
   0.9667386  -1.4121602 ]
 [ 2.7112627  -1.0778131   0.69413143 -0.14870553  0.51810503  0.3507294
  -0.82285154  0.80680597]
 [-0.13484643 -1.868895    0.01032507 -0.13787179 -0.12893449  0.03143518
   0.2687057   0.13212144]], shape=(3, 8), dtype=float32)
y = tf.Tensor(
[[1.819]
 [3.674]
 [0.954]], shape=(3, 1), dtype=float32)

X = tf.Tensor(
[[ 0.09031774  0.9789995   0.1327582  -0.13753782 -0.23388447  0.10211545
   0.97610843 -1.4121602 ]
 [ 0.05218809 -2.0271113   0.2940109  -0.02403445  0.16218767 -0.02844518
   1.4117942  -0.93737936]
 [-0.672276    0.02970133 -0.76922584 -0.15086786  0.4962024  -0.02741998
  -0.7853724   0.77182245]], shape=(3, 8), dtype=float32)
y = tf.Tensor(
[[2.725]
 [1.205]
 [1.625]], shape=(3, 1), dtype=float32)



## 프리페치 (`prefetch()`)
> 하나의 배치가 미리 준비되도록 준비시키는 함수 

### 특징
- 하나의 함수가 데이터셋의 다음 배치를 미리 준비해 두어, 모델이 현재 배치를 처리하는 동안 다음 배치를 로드
- 성능 향상에 도움이 됨 
- 멀티스레드로 데이터 적재 및 전처리 하면 CPU를 여러 개 활용해도 GPU를 100% 활용 가능

### cache()
- 데이터셋이 메모리에 들어갈 정도로 작은 경우 훈련속도를 높일 수 있는 방법 
- 샘플을 한 번만 읽고 전처리하지만 에포크마다 다르게 셔플링 후 다음 배치도 미리 준비 

- 추가 : Dataset 클래스 매소드 설명

In [67]:
for m in dir(tf.data.Dataset):
    if not (m.startswith("_") or m.endswith("_")):
        func = getattr(tf.data.Dataset, m)
        if hasattr(func, "__doc__"):
            print("● {:21s}{}".format(m + "()", func.__doc__.split("\n")[0]))

● apply()              Applies a transformation function to this dataset.
● as_numpy_iterator()  Returns an iterator which converts all elements of the dataset to numpy.
● batch()              Combines consecutive elements of this dataset into batches.
● bucket_by_sequence_length()A transformation that buckets elements in a `Dataset` by length.
● cache()              Caches the elements in this dataset.
● cardinality()        Returns the cardinality of the dataset, if known.
● choose_from_datasets()Creates a dataset that deterministically chooses elements from `datasets`.
● concatenate()        Creates a `Dataset` by concatenating the given dataset with this dataset.
● counter()            Creates a `Dataset` that counts from `start` in steps of size `step`.
● element_spec()       The type specification of an element of this dataset.
● enumerate()          Enumerates the elements of this dataset.
● filter()             Filters this dataset according to `predicate`.
● fingerprint()     

## 케라스와 데이터셋 사용

### 매소드 
- `csv_reader_dataset()` : CSV 파일을 읽어와 데이터셋으로 변환하는 함수
  - 훈련 세트는 각 에포크 마다 셔플링

- `fit()` : 모델 훈련 
- `evaluate()` : 훈련 점수 출력
- `predicet()` : 예측 실행 

In [68]:
train_set = csv_reader_dataset(train_filepaths)
valid_set = csv_reader_dataset(valid_filepaths)
test_set = csv_reader_dataset(test_filepaths)

- 모델 훈련

In [69]:
### 초기화
tf.keras.backend.clear_session()
tf.random.set_seed(42)

### 데이터셋 훈련 
model = tf.keras.Sequential([
    tf.keras.layers.Dense(30, activation="relu", kernel_initializer="he_normal",
                          input_shape=X_train.shape[1:]),
    tf.keras.layers.Dense(1),
])
model.compile(loss="mse", optimizer="sgd")
model.fit(train_set, validation_data=valid_set, epochs=5)


Epoch 1/5


c:\ProgramData\miniconda3\envs\ml_basic\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    326/Unknown 1s 1ms/step - loss: 1.4069

c:\ProgramData\miniconda3\envs\ml_basic\Lib\site-packages\keras\src\trainers\epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3567 - val_loss: 137.8062
Epoch 2/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3777 - val_loss: 3.0133
Epoch 3/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6450 - val_loss: 23.8511
Epoch 4/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5332 - val_loss: 17.3217
Epoch 5/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8027 - val_loss: 0.3973


- 모델 평가 및 예측

In [70]:
test_mse = model.evaluate(test_set)

### 예시 데이터 
new_set = test_set.take(3)

### 예측
y_pred = model.predict(new_set)

162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 910us/step - loss: 0.3806
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


c:\ProgramData\miniconda3\envs\ml_basic\Lib\site-packages\keras\src\trainers\epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


-  훈련을 위한 옵티마이저 및 손실함수 생성
-  레이블이 없는 데이터를 위해 넘파이 배열을 넘겨줌

In [72]:
#  훈련을 위한 옵티마이저 및 손실 함수를 정의
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
loss_fn = tf.keras.losses.MeanSquaredError()

n_epochs = 5
for epoch in range(n_epochs):
    for X_batch, y_batch in train_set:
        #경사 하강법 스텝 하나를 수행
        print("\rEpoch {}/{}".format(epoch + 1, n_epochs), end="")
        with tf.GradientTape() as tape:
            y_pred = model(X_batch)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

Epoch 5/5

- 에포크 동안 모델 훈련을 하는 함수 생성 
- `steps_per_excution`으로 `fit()`에서 처리할 배치 수 정의 가능 
- `on_batch_*()` : 배치 처리 후 콜백

In [74]:
@tf.function
def train_one_epoch(model, optimizer, loss_fn, train_set):
    for X_batch, y_batch in train_set:
        with tf.GradientTape() as tape:
            y_pred = model(X_batch)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
loss_fn = tf.keras.losses.MeanSquaredError()
for epoch in range(n_epochs):
    print("\rEpoch {}/{}".format(epoch + 1, n_epochs), end="")
    train_one_epoch(model, optimizer, loss_fn, train_set)

Epoch 5/5

-----

# 2. TFRecord 

## 개요
> 크기가 다른 연속된 이진레코드를 저장하는 단순 이진 포맷

- 구성 : 레코드 길이, CRC체크섬, 실제데이터 
- CSV 파일을 이용한 데이터 처리는 효율적이지 못함 
- 대용량 데이터 저장 및 읽기를 위한 포맷은 `TFRecord` 사용

### TFRecord 생성
- `tf.data.TFRecordWriter`를 사용하여 TFRecord 파일 생성

- 예시 : TFRecord 파일 생성

In [75]:
with tf.io.TFRecordWriter("my_data.tfrecord") as f:
    f.write(b"This is the first record")
    f.write(b"And this is the second record")

### TFRecord 파일 읽기
- `tf.data.TFRecordDataset` 를 이용한 읽기 

In [76]:
filepaths = ["my_data.tfrecord"]
dataset = tf.data.TFRecordDataset(filepaths)
for item in dataset:
    print(item)

tf.Tensor(b'This is the first record', shape=(), dtype=string)
tf.Tensor(b'And this is the second record', shape=(), dtype=string)


## 압축된 TFRecord 파일 생성

### 압축 TFRecord 생성
  - `option` : 매개 변수로 압축된 TFRecord 파일을 생성

In [ ]:
options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter("my_compressed.tfrecord", options) as f:
    f.write(b"Compress, compress, compress!")

### TFRecord 파일 읽기 
- 파일 압축 형식을 지정해야 읽을 수 있음 

In [78]:
dataset = tf.data.TFRecordDataset(["my_compressed.tfrecord"],
                                  compression_type="GZIP")

for item in dataset:
    print(item)

tf.Tensor(b'Compress, compress, compress!', shape=(), dtype=string)


## 프로토콜 버퍼 개요

> 구글에서 개발한 이식성과 확장성이 좋은 이진 포맷 

- gRPC에 사용됨

- 설정 방법

In [79]:
%%writefile person.proto
syntax = "proto3";
message Person {
    string name = 1;
    int32 id = 2;
    repeated string email = 3;
}

Writing person.proto


In [ ]:
!protoc person.proto --python_out=. --descriptor_set_out=person.desc --include_imports


- 일반적으로 텐서플로에서 사용할 프로토콜 버퍼 정의는 이미 컴파일 됨
  - 텐서플로 안에 파이썬 클래스에 포함됨 
  - 따라서 `protoc` 필요 없음 

In [84]:
from person_pb2 import Person 

### Person 생성
person = Person(name="Al", id=123, email=["a@b.com"])  
print("Person 출력 : ",person)  

### 필드 읽기 
print("이름 : ", person.name)

### 필드 수정 
person.name = "Alice"
print("수정된 이름 : ", person.name)

# 반복되는 필드는 배열처럼 액세스 가능
print("이메일 : ", person.email[0])

# person을 바이트 문자열로 직렬화
serialized = person.SerializeToString()
print("직렬화된 Person : ", serialized)

# 새로운 Person 만들기
person2 = Person()  
person2.ParseFromString(serialized) 

print("동일한지 확인 : ",person == person2)

Person 출력 :  name: "Al"
id: 123
email: "a@b.com"

이름 :  Al
수정된 이름 :  Alice
이메일 :  a@b.com
직렬화된 Person :  b'\n\x05Alice\x10{\x1a\x07a@b.com'
동일한지 확인 :  True


----

## 텐서플로 프로토콜 버퍼

> TFRecord 파일에서 사용하는 전형적인 주요 프로토콜 버퍼는 데이터셋에 있는 하나의 샘플을 표현하는 `Example` 프로토콜 버퍼

- 이름을 가진 특성의 리스트를 가지고 있음 
- `[packed=true]` : 효율적인 인코딩을 위한 반복적인 수치 필드 사용 
- `Feature` : 특성 이름 + 특성값 매핑한 딕셔너리
- `Example` : `Feature` 객체 하나를 가짐 

- `Example` 객체 생성

In [86]:
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

person_example = Example(
    features=Features(
        feature={
            "name": Feature(bytes_list=BytesList(value=[b"Alice"])),
            "id": Feature(int64_list=Int64List(value=[123])),
            "emails": Feature(bytes_list=BytesList(value=[b"a@b.com",
                                                          b"c@d.com"]))
        }))

In [87]:
with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    for _ in range(5):
        f.write(person_example.SerializeToString())

- `SerializeToString()` : 직렬화 및 TFRecord 파일 저장 

In [88]:
with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    for _ in range(5):
        f.write(person_example.SerializeToString())

## Example 프로토콜 버퍼 읽고 파싱

- `TFRecordDataset` : TFRecord 파일에서 데이터를 읽고 파싱하는 데 사용
- `tf.io.parse_single_example` : TFRecord 파일에서 단일 예제를 파싱하는 데 사용
- `tf.io.FixedLenFeature` : 고정 길이의 특성을 정의하는 데 사용
- `tf.io.VarLenFeature` : 가변 길이의 특성을 정의하는 데 사용

In [89]:
feature_description = {
    "name": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails": tf.io.VarLenFeature(tf.string),
}

def parse(serialized_example):
    return tf.io.parse_single_example(serialized_example, feature_description)

dataset = tf.data.TFRecordDataset(["my_contacts.tfrecord"]).map(parse)
for parsed_example in dataset:
    print(parsed_example)

{'emails': SparseTensor(indices=tf.Tensor(
[[0]
 [1]], shape=(2, 1), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com'], shape=(2,), dtype=string), dense_shape=tf.Tensor([2], shape=(1,), dtype=int64)), 'id': <tf.Tensor: shape=(), dtype=int64, numpy=123>, 'name': <tf.Tensor: shape=(), dtype=string, numpy=b'Alice'>}
{'emails': SparseTensor(indices=tf.Tensor(
[[0]
 [1]], shape=(2, 1), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com'], shape=(2,), dtype=string), dense_shape=tf.Tensor([2], shape=(1,), dtype=int64)), 'id': <tf.Tensor: shape=(), dtype=int64, numpy=123>, 'name': <tf.Tensor: shape=(), dtype=string, numpy=b'Alice'>}
{'emails': SparseTensor(indices=tf.Tensor(
[[0]
 [1]], shape=(2, 1), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com'], shape=(2,), dtype=string), dense_shape=tf.Tensor([2], shape=(1,), dtype=int64)), 'id': <tf.Tensor: shape=(), dtype=int64, numpy=123>, 'name': <tf.Tensor: shape=(), dtype=string, numpy=b'Alice'>}
{'emails': SparseTensor(indices=tf.Tenso

- 배치 단위로 파싱 : `tf.io.parse_example()`

In [90]:
tf.sparse.to_dense(parsed_example["emails"], default_value=b"")

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [91]:
parsed_example["emails"].values

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [92]:
def parse(serialized_examples):
    return tf.io.parse_example(serialized_examples, feature_description)

dataset = tf.data.TFRecordDataset(["my_contacts.tfrecord"]).batch(2).map(parse)
for parsed_examples in dataset:
    print(parsed_examples)  # two examples at a time

{'emails': SparseTensor(indices=tf.Tensor(
[[0 0]
 [0 1]
 [1 0]
 [1 1]], shape=(4, 2), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com' b'a@b.com' b'c@d.com'], shape=(4,), dtype=string), dense_shape=tf.Tensor([2 2], shape=(2,), dtype=int64)), 'id': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([123, 123])>, 'name': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'Alice', b'Alice'], dtype=object)>}
{'emails': SparseTensor(indices=tf.Tensor(
[[0 0]
 [0 1]
 [1 0]
 [1 1]], shape=(4, 2), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com' b'a@b.com' b'c@d.com'], shape=(4,), dtype=string), dense_shape=tf.Tensor([2 2], shape=(2,), dtype=int64)), 'id': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([123, 123])>, 'name': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'Alice', b'Alice'], dtype=object)>}
{'emails': SparseTensor(indices=tf.Tensor(
[[0 0]
 [0 1]], shape=(2, 2), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com'], shape=(2,), dtype=string), dense_shape=tf.Ten

In [93]:
parsed_examples

{'emails': SparseTensor(indices=tf.Tensor(
 [[0 0]
  [0 1]], shape=(2, 2), dtype=int64), values=tf.Tensor([b'a@b.com' b'c@d.com'], shape=(2,), dtype=string), dense_shape=tf.Tensor([1 2], shape=(2,), dtype=int64)),
 'id': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([123])>,
 'name': <tf.Tensor: shape=(1,), dtype=string, numpy=array([b'Alice'], dtype=object)>}

## SequenceExample 프로토콜 버퍼로 리스트의 리스트 다루기 

In [94]:
from tensorflow.train import FeatureList, FeatureLists, SequenceExample

context = Features(feature={
    "author_id": Feature(int64_list=Int64List(value=[123])),
    "title": Feature(bytes_list=BytesList(value=[b"A", b"desert", b"place", b"."])),
    "pub_date": Feature(int64_list=Int64List(value=[1623, 12, 25]))
})

content = [["When", "shall", "we", "three", "meet", "again", "?"],
           ["In", "thunder", ",", "lightning", ",", "or", "in", "rain", "?"]]
comments = [["When", "the", "hurlyburly", "'s", "done", "."],
            ["When", "the", "battle", "'s", "lost", "and", "won", "."]]

def words_to_feature(words):
    return Feature(bytes_list=BytesList(value=[word.encode("utf-8")
                                               for word in words]))

content_features = [words_to_feature(sentence) for sentence in content]
comments_features = [words_to_feature(comment) for comment in comments]

sequence_example = SequenceExample(
    context=context,
    feature_lists=FeatureLists(feature_list={
        "content": FeatureList(feature=content_features),
        "comments": FeatureList(feature=comments_features)
    }))

In [95]:
sequence_example

context {
  feature {
    key: "title"
    value {
      bytes_list {
        value: "A"
        value: "desert"
        value: "place"
        value: "."
      }
    }
  }
  feature {
    key: "pub_date"
    value {
      int64_list {
        value: 1623
        value: 12
        value: 25
      }
    }
  }
  feature {
    key: "author_id"
    value {
      int64_list {
        value: 123
      }
    }
  }
}
feature_lists {
  feature_list {
    key: "content"
    value {
      feature {
        bytes_list {
          value: "When"
          value: "shall"
          value: "we"
          value: "three"
          value: "meet"
          value: "again"
          value: "?"
        }
      }
      feature {
        bytes_list {
          value: "In"
          value: "thunder"
          value: ","
          value: "lightning"
          value: ","
          value: "or"
          value: "in"
          value: "rain"
          value: "?"
        }
      }
    }
  }
  feature_list {
    key: "c

# 3. 케라스의 전처리층 

## 개요 

### 전처리 방법 
- numpy 혹은 pandas 등의 라이브러리에서 미리 전처리 수행 
- 텐서플로우의 map() 매서드를 이용하여 모든 원소 전처리 
- 모델 내부 전처리 층을 직접 포함해 훈련중 모든 입력을 즉시 전처리 

## Nomalization 층 

> 입력특성을 표준화 하는 층 

- 층 생성시 평균 및 분산 지정 가능 
- 모델 훈련 전 `adpt()` 를 통해 훈션 세트를 전달하여 특성 및 평균 분산 계산가능 

### 특징
#### 장점
- Nomalization 층에 포함하기 때문에 정규화를 고려하지 않고 모델 배포 가능 
- 전처리 불일치의 위험이 제거됨 
  - 전처리 코드를 별도로 유지하다가 다른 쪽이 업데이트 되는 등의 문제 해소 

#### 단점 
-  훈련 속도를 느리게 만듦 
- 에포크 마다 정규화가 이뤄짐 
- 해결 방법 
  - 훈련 전에 전체 훈련세트 딱 한번만 전처리 되도록 독립적으로 사용
  - 단, 모델을 제품에 배포했을 때, 입력 전처리가 불가능 
  - Normalizaion층과 훈련된 모델을 포함하는 방식으로 해결 

- 예시 : Nomalization을 통한 전처리

In [96]:
tf.random.set_seed(42)  

### Nomalization 층
norm_layer = tf.keras.layers.Normalization()
model = tf.keras.models.Sequential([
    norm_layer,
    tf.keras.layers.Dense(1)
])
model.compile(loss="mse", optimizer=tf.keras.optimizers.SGD(learning_rate=2e-3))

 # 모든 특성의 평균과 분산을 계산
norm_layer.adapt(X_train) 
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=5)

Epoch 1/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.7771 - val_loss: 0.7555
Epoch 2/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7368 - val_loss: 0.6242
Epoch 3/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5826 - val_loss: 0.6051
Epoch 4/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5693 - val_loss: 0.6059
Epoch 5/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5653 - val_loss: 0.6027


- 예시 : 딱 한 번만 전체 훈련세트를 훈련 시키는 방법

In [97]:
norm_layer = tf.keras.layers.Normalization()
norm_layer.adapt(X_train)
X_train_scaled = norm_layer(X_train)
X_valid_scaled = norm_layer(X_valid)

  - 조정된 데이터로 모델 훈련 (단, Nomlizaion 층 미사용)

In [98]:
tf.random.set_seed(42) 

### Nomalizaion 층 미포함 
model = tf.keras.models.Sequential([tf.keras.layers.Dense(1)])
model.compile(loss="mse", optimizer=tf.keras.optimizers.SGD(learning_rate=2e-3))
model.fit(X_train_scaled, y_train, epochs=5,
          validation_data=(X_valid_scaled, y_valid))

Epoch 1/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 4.6164 - val_loss: 1.0905
Epoch 2/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9572 - val_loss: 0.9241
Epoch 3/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7280 - val_loss: 0.8056
Epoch 4/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6856 - val_loss: 0.7636
Epoch 5/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6592 - val_loss: 0.7299


- 예시 : 제품에서  Nomalization 층을 통한 전처리 추가 

In [99]:
### 새로운 입력에 대한 전처리 추가
final_model = tf.keras.Sequential([norm_layer, model])

### 정규화되지 않은 샘플 추가
X_new = X_test[:3]  

### 모델 예측 
y_pred = final_model(X_new)  

### 예측 출력
print("y_pred : ", y_pred) 

y_pred :  tf.Tensor(
[[0.98155093]
 [1.5904713 ]
 [2.3820438 ]], shape=(3, 1), dtype=float32)


### 데이터API와 함께 사용 
- `adapt()` : 매서드로 객체 전달 가능 
- `map()` :  케라스 전처리 층 적용 가능 

- 예시 : `adapt()` 매소드를 이용한 Nomalization층 데이터 셋에 있는 각 배치 특성에 적용 

In [102]:
### map을 이용한 norm_layer 적용용 데이터 생성 
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(5)

dataset = dataset.map(lambda X, y: (norm_layer(X), y))

### 첫번째 배치 표시
list(dataset.take(1)) 

[(<tf.Tensor: shape=(5, 8), dtype=float32, numpy=
  array([[-0.19397889, -1.0778131 , -0.9433854 ,  0.01485314,  0.02073333,
          -0.57291627,  0.9292612 , -1.4221537 ],
         [ 0.7519831 , -1.868895  ,  0.40547806, -0.23327684,  1.8614649 ,
           0.20516537, -0.9165477 ,  1.0966716 ],
         [-0.41469118,  0.02970133,  0.8180882 ,  1.0567837 , -0.0878671 ,
          -0.2998328 ,  1.3087282 , -1.697027  ],
         [ 1.7188957 , -1.3151377 ,  0.32664376, -0.2195511 , -0.33792186,
          -0.11146631, -0.9821345 ,  0.94174504],
         [-0.9620722 , -1.2360295 , -0.05625783, -0.03124396,  1.7090592 ,
          -0.30256987, -0.80411196,  1.3265638 ]], dtype=float32)>,
  <tf.Tensor: shape=(5, 1), dtype=float64, numpy=
  array([[1.442],
         [1.687],
         [1.621],
         [2.621],
         [0.956]])>)]

### 사용자 정의 케라스 층 정의 

In [104]:
class MyNormalization(tf.keras.layers.Layer):
    def adapt(self, X):
        self.mean_ = np.mean(X, axis=0, keepdims=True)
        self.std_ = np.std(X, axis=0, keepdims=True)

    def call(self, inputs):
        eps = tf.keras.backend.epsilon()  # 0 나눗셈 방지
        return (inputs - self.mean_) / (self.std_ + eps)
    
my_norm_layer = MyNormalization()
my_norm_layer.adapt(X_train)
X_train_scaled = my_norm_layer(X_train)

## Discretization 층 

### 특징 
- 값 범위를 범주로 매핑하여 수치 특성을 범주형 특성으로 변환 
- 다중모드 분포를 가진 특성이나 매우 비션형적인 특성에 유용 
- `adapt()`를 통해 적절한 구간 경계 찾기 가능 
- `num_bins`를 지정하면 구간 경계 구간 숫자 설정 가능 

- 예시 : `age` 특성을 나누고 범주로 매핑 

In [105]:
age = tf.constant([[10.], [93.], [57.], [18.], [37.], [5.]])
discretize_layer = tf.keras.layers.Discretization(bin_boundaries=[18., 50.])
age_categories = discretize_layer(age)
age_categories

<tf.Tensor: shape=(6, 1), dtype=int64, numpy=
array([[0],
       [2],
       [2],
       [1],
       [1],
       [0]])>

- 구간 경계 선정

In [106]:
discretize_layer = tf.keras.layers.Discretization(num_bins=3)
discretize_layer.adapt(age)
age_categories = discretize_layer(age)
age_categories

<tf.Tensor: shape=(6, 1), dtype=int64, numpy=
array([[1],
       [2],
       [2],
       [1],
       [2],
       [0]])>

## CategoryEncoding 층

- 범주의 수가 적을 때 `원-핫 인코딩` 이용하는 방법 

- 예시 : 앞 예시의 `원-핫 인코딩` 적용

In [107]:
onehot_layer = tf.keras.layers.CategoryEncoding(num_tokens=3)
onehot_layer(age_categories)

<tf.Tensor: shape=(6, 3), dtype=float32, numpy=
array([[0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]], dtype=float32)>

- 동시에 한 개 이상 범주형 특성 인코딩 시 자동으로 `멀티-핫 인코딩` 수행 
    - 입력 특성에 있는 범주에 있는 위치 마다 출력 텐서의 값이 1

In [109]:
two_age_categories = np.array([[1, 0], [2, 2], [2, 0]])
onehot_layer(two_age_categories)

<tf.Tensor: shape=(3, 3), dtype=float32, numpy=
array([[1., 1., 0.],
       [0., 0., 2.],
       [1., 0., 1.]], dtype=float32)>

### 단점 
- 멀티-핫 인코딩과 카운트 인코딩은 범주를 활성화 한 특성을 알 수 없음 
  - 따라서 정보 손실의 발생 
  - 해결책 : 특성마다 별도의 원-핫 인코딩 출력 
    - 모델에 주입할 특성 개수를 증가시킴 -> 모델 파라미터가 더 필요해짐

- 예시 : 별도 원-핫 인코딩을 작성하여 정보손실 방지

In [110]:
onehot_layer = tf.keras.layers.CategoryEncoding(num_tokens=3, output_mode="count")
onehot_layer(two_age_categories)

<tf.Tensor: shape=(3, 3), dtype=float32, numpy=
array([[1., 1., 0.],
       [0., 0., 2.],
       [1., 0., 1.]], dtype=float32)>

In [111]:
onehot_layer = tf.keras.layers.CategoryEncoding(num_tokens=3 + 3)
onehot_layer(two_age_categories + [0, 3]) 

<tf.Tensor: shape=(3, 6), dtype=float32, numpy=
array([[0., 1., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 1.],
       [0., 0., 1., 1., 0., 0.]], dtype=float32)>

## StringLookup층 

> 텍스트로된 범주형 특성 처리를 위한 층

- `StringLookup` 층 생성
- -> `adapt()` 매서드에 데이터 전달 
- -> 3개의 범주 찾음 
- -> 몇 개 도시 인코딩 (`output_mode="one_hot"` 지정 시 원핫벡터 출력)

- 예시 : `cities` 특성 `원-핫 인코딩`

In [112]:
cities = ["Auckland", "Paris", "Paris", "San Francisco"]
str_lookup_layer = tf.keras.layers.StringLookup()
str_lookup_layer.adapt(cities)
str_lookup_layer([["Paris"], ["Auckland"], ["Auckland"], ["Montreal"]])

<tf.Tensor: shape=(4, 1), dtype=int64, numpy=
array([[1],
       [3],
       [3],
       [0]])>

- 예시 : `cities` 특성 `원-핫 인코딩`

In [114]:
str_lookup_layer = tf.keras.layers.StringLookup(output_mode="one_hot")
str_lookup_layer.adapt(cities)
str_lookup_layer([["Paris"], ["Auckland"], ["Auckland"], ["Montreal"]])

<tf.Tensor: shape=(4, 4), dtype=int64, numpy=
array([[0, 1, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 0, 1],
       [1, 0, 0, 0]])>

- 예시 : `cities` 특성 `IntegerLookup`층 이용

In [115]:
ids = [123, 456, 789]
int_lookup_layer = tf.keras.layers.IntegerLookup()
int_lookup_layer.adapt(ids)
int_lookup_layer([[123], [456], [123], [111]])

<tf.Tensor: shape=(4, 1), dtype=int64, numpy=
array([[3],
       [2],
       [3],
       [0]])>

## Hashing 층 
### 개요
- `해싱 충돌` : 서로 다른 입력이 동일한 해시 값을 가질 때 발생
  - 해결책 : OOV 버킷 수를 늘리는 것 
  - 문제점 : 범주의 전체 개수가 증가 -> 필요 메모리 증가 -> 파라미터수 증가 
  - 범주를 랜덤하게 버킷에 매핑 -> `해싱 트릭`

### 정의 
> 범주마다 해시 계산 후, 버킷 개수로 나눈 나머지를 구함

### 특징 
- 재현 가능한 랜덤 -> 안정적임 

#### 장점 
- `adapt()` 매서드 호출 필요 없음 
- 데이터셋이 너무 커서 외부 메모리를 이용하는 경우 

#### 단점 
- 해싱충돌이 일어남 

- 예시 : Hashing을 통한 `cities` 특성 인코딩

In [116]:
hashing_layer = tf.keras.layers.Hashing(num_bins=10)
hashing_layer([["Paris"], ["Tokyo"], ["Auckland"], ["Montreal"]])

<tf.Tensor: shape=(4, 1), dtype=int64, numpy=
array([[0],
       [1],
       [9],
       [1]])>